In [1]:
import geopandas as gpd, openrouteservice, shapely, folium, matplotlib.pyplot as mpl, mapclassify, time, pandas as pd
from shapely.geometry import shape, Point
from requests.exceptions import ConnectionError

#API KEY
from config import OPENROUTESERVICE_API_KEY

In [2]:
block_groups = gpd.read_file("C:/Users/user00/Desktop/Ridership Project Data/Project 2 Data/tl_2023_34_bg/tl_2023_34_bg.shp")
Data = pd.read_csv("Stop_Data.csv")
nj_census_data = pd.read_csv("nj_census_data.csv")

client = openrouteservice.Client(key=OPENROUTESERVICE_API_KEY)

## Isochrone Generation and Export Using OpenRouteService

This section generates 15-minute walking and 10-minute driving isochrones around NJ Transit stations using the OpenRouteService (ORS) API. These isochrone polygons define areas reachable within a specific travel time from each station, and they are later used for spatial analysis with demographic data.

---

### Steps Overview

1. **Load Required Data**
   - `block_groups`: New Jersey block group shapefile for spatial intersection.
   - `Data`: Station data containing latitude, longitude, and station names.
   - `nj_census_data`: ACS-based demographic data for NJ block groups.

2. **Initialize the ORS Client**
   - The OpenRouteService client is created using your API key.
   - This enables access to walking and driving isochrone endpoints.

3. **Define Isochrone Request Function**
   - `get_isochrone(lat, lon, minutes, profile)`:
     - Accepts station coordinates, time range in minutes, and travel mode (`foot-walking` or `driving-car`).
     - Retries up to 3 times if a `ConnectionError` occurs.
     - Returns a Shapely polygon geometry.

4. **Generate Isochrones in Batches**
   - Stations are processed in batches of 9 (yielding 18 isochrones per batch) to respect ORS's 20 requests/minute rate limit.
   - For each station in the batch:
     - A 15-minute **walking** isochrone is generated.
     - A 10-minute **driving** isochrone is generated.
   - Results are appended to the `isochrones` list as dictionaries containing:
     - Station name
     - Walking isochrone geometry
     - Driving isochrone geometry
   - The script pauses for 65 seconds between batches to remain compliant with rate limits.

5. **Create and Export GeoDataFrames**
   - `walking_isochrones_gdf`: A `GeoDataFrame` using the walking isochrones as the main geometry.
     - Saved to `isochrones.gpkg` as layer `"walking_isochrones"`.
     - Can be visualized interactively using `.explore()`.
   - `driving_isochrones_gdf`: A `GeoDataFrame` using the driving isochrones as the main geometry.
     - Saved to the same GeoPackage under layer `"driving_isochrones"`.

---

These GeoPackages serve as persistent spatial datasets that can be later used for visualization or spatial joins with ACS census block group data to compute demographic statistics within each isochrone.


In [4]:
def get_isochrone(lat, lon, minutes, profile='foot-walking'):
    attempt = 0
    max_attempts = 3
    while attempt < max_attempts:
        try:
            # Log attempt and profile
            print(f"Attempting to fetch {profile} isochrone for {minutes} minutes (attempt {attempt + 1})")
            # Request isochrone from ORS
            isochrone = client.isochrones(
                locations=[[lon, lat]],
                profile=profile,
                range=[minutes * 60],  # time in seconds
                units='m'  # meters
            )
            return shape(isochrone['features'][0]['geometry'])
        except ConnectionError as e:
            print(f"ConnectionError: {e}. Retrying in 5 seconds...")
            time.sleep(5)
            attempt += 1
    print(f"Failed to fetch isochrone after {max_attempts} attempts.")
    return None

# Load your station data
stations = Data

# Create GeoDataFrame to hold walking and driving isochrones
isochrones = []

# Process stations in batches to respect API limits
batch_size = 9  # 10 stations = 20 isochrones per batch
for i in range(0, len(stations), batch_size):
    batch = stations[i:i + batch_size]

    for index, row in batch.iterrows():
        lat = row['stop_lat']  # Adjusted for 'stop_lat' column
        lon = row['stop_lon']  # Adjusted for 'stop_lon' column
        station_name = row['Official NJT Station Name']

        # Generate 15 min walking isochrone
        print(f"Processing 15 min walking isochrone for {station_name}...")
        walking_isochrone = get_isochrone(lat, lon, 15, profile='foot-walking')

        # Generate 10 min driving isochrone
        print(f"Processing 10 min driving isochrone for {station_name}...")
        driving_isochrone = get_isochrone(lat, lon, 10, profile='driving-car')

        # Append isochrones to the list if successfully retrieved
        if walking_isochrone and driving_isochrone:
            isochrones.append({
                'station_name': station_name,
                'walking_isochrone': walking_isochrone,
                'driving_isochrone': driving_isochrone
            })

    # Pause for 65 seconds to respect the rate limit (20 isochrones per minute)
    print(f"Processed batch {i // batch_size + 1}. Waiting for 65 seconds to respect rate limit...")
    time.sleep(65)

Processing 15 min walking isochrone for METROPARK...
Attempting to fetch foot-walking isochrone for 15 minutes (attempt 1)
Processing 10 min driving isochrone for METROPARK...
Attempting to fetch driving-car isochrone for 10 minutes (attempt 1)
Processing 15 min walking isochrone for PRINCETON JCT....
Attempting to fetch foot-walking isochrone for 15 minutes (attempt 1)
Processing 10 min driving isochrone for PRINCETON JCT....
Attempting to fetch driving-car isochrone for 10 minutes (attempt 1)
Processing 15 min walking isochrone for HAMILTON...
Attempting to fetch foot-walking isochrone for 15 minutes (attempt 1)
Processing 10 min driving isochrone for HAMILTON...
Attempting to fetch driving-car isochrone for 10 minutes (attempt 1)
Processing 15 min walking isochrone for NEW BRUNSWICK...
Attempting to fetch foot-walking isochrone for 15 minutes (attempt 1)
Processing 10 min driving isochrone for NEW BRUNSWICK...
Attempting to fetch driving-car isochrone for 10 minutes (attempt 1)
Proc

In [5]:
# Create GeoDataFrame for walking isochrones
walking_isochrones_gdf = gpd.GeoDataFrame(
    isochrones,
    geometry='walking_isochrone',  # Use walking isochrone as the main geometry
    crs='EPSG:4326'
)

# Save walking isochrones to GeoPackage
walking_isochrones_gdf.to_file('isochrones.gpkg', layer='walking_isochrones', driver="GPKG")

# Visualize walking isochrones
walking_isochrones_gdf.explore(legend=True)

# Create GeoDataFrame for driving isochrones
driving_isochrones_gdf = gpd.GeoDataFrame(
    isochrones,
    geometry='driving_isochrone',  # Use driving isochrone as the main geometry
    crs='EPSG:4326'
)

# Save driving isochrones to the same GeoPackage as a different layer
driving_isochrones_gdf.to_file('isochrones.gpkg', layer='driving_isochrones', driver="GPKG")

In [6]:
#Reloading isochrones.gpkg to check if it worked.

# Read the walking isochrones layer from the GeoPackage
walking_isochrones_gdf = gpd.read_file('C:/Users/user00/PycharmProjects/NJT Ridership Project 2/isochrones.gpkg', layer='walking_isochrones')

# Read the driving isochrones layer from the GeoPackage
driving_isochrones_gdf = gpd.read_file('C:/Users/user00/PycharmProjects/NJT Ridership Project 2/isochrones.gpkg', layer='driving_isochrones')

## Intersecting Walking Isochrones with Census Block Groups

This section performs a spatial intersection between 15-minute walking isochrones and U.S. Census block groups in New Jersey to compute localized demographic metrics within accessible areas around NJ Transit stations.

---

### Steps Overview

1. **Prepare Data for Merging**
   - Ensure the `GEOID` field in `nj_census_data` is of type `str` to match the corresponding field in the shapefile.
   - Merge the census data (`nj_census_data`) with the geographic block group shapefile (`block_groups`) using the `GEOID` key.
   - Save the merged `GeoDataFrame` to a GeoPackage file named `merged_block_groups.gpkg`.

2. **Compute Area and Population Density**
   - Convert the land area (`ALAND`) from square meters to square kilometers:
     `ALAND_km2 = ALAND / 1,000,000`
   - Compute population density as:
     `Population Density = Total Population / Area in km²`

3. **Coordinate Reference System (CRS) Alignment**
   - Ensure both the block group polygons and isochrone polygons are in the same CRS (`EPSG:4326`) for accurate spatial operations.
   - Use `.to_crs('EPSG:4326')` to standardize both GeoDataFrames.

4. **Perform Spatial Intersection**
   - **Option 1: `gpd.overlay()`**
     - Performs an exact geometric intersection and creates new geometries for the overlapping areas.
   - **Option 2: `gpd.sjoin()`**
     - Performs a spatial join by selecting all block groups that intersect with a walking isochrone polygon (faster but less precise geometrically).

5. **Save the Intersection Results**
   - Export the intersected `GeoDataFrame` as a new layer called `walking_isochrone_intersections` in `intersection_results.gpkg`.

---

This intersection allows us to identify and analyze the demographic characteristics of populations within walking distance (15 minutes) of each NJ Transit rail station. These enriched features can be used for ridership modeling and spatial accessibility analysis.


In [7]:
nj_census_data['GEOID'] = nj_census_data['GEOID'].astype(str)
merged_block_groups = block_groups.merge(nj_census_data, on='GEOID')
merged_block_groups.to_file('merged_block_groups.gpkg', driver='GPKG')
merged_block_groups['ALAND_km2'] = merged_block_groups['ALAND'] / 1000000
merged_block_groups['Popoulation_Density'] = merged_block_groups['Total_Population'] / merged_block_groups['ALAND_km2']

In [8]:
# Ensure both GeoDataFrames are in the same CRS (EPSG:4326 in this case)
walking_isochrones_gdf = walking_isochrones_gdf.to_crs('EPSG:4326')
merged_block_groups = merged_block_groups.to_crs('EPSG:4326')

# Perform the spatial intersection using gpd.overlay() or gpd.sjoin()
# Option 1: Using overlay (for exact geometric intersections)
intersection_gdf = gpd.overlay(merged_block_groups, walking_isochrones_gdf, how='intersection', keep_geom_type=False)

# Option 2: Using spatial join (to just join features that intersect)
intersection_gdf = gpd.sjoin(merged_block_groups, walking_isochrones_gdf, how='inner', predicate='intersects')

# Save the result as a new GeoPackage or shapefile
intersection_gdf.to_file('intersection_results.gpkg', layer='walking_isochrone_intersections', driver='GPKG')

In [9]:
#Reloading intersection_results.gpkg to to check if it worked.
intersection_gdf = gpd.read_file('C:/Users/user00/PycharmProjects/NJT Ridership Project 2/intersection_results.gpkg',
                                 layer='walking_isochrone_intersections')
intersection_gdf.head(10)

,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOID,GEOIDFQ,NAMELSAD,MTFCC,FUNCSTAT,ALAND,...,Unemployment_Rate,Percent_Public_Transit_Users,Labor_Force_Participation_Rate,Percent_Bach_Degree,ALAND_km2,Popoulation_Density,index_right,station_name,driving_isochrone,geometry
0,34,023,003401,4,340230034014,1500000US340230034014,Block Group 4,G5030,S,404027,...,1.291990,0.000000,68.495575,31.546392,0.404027,1527.125662,20,WOODBRIDGE,"POLYGON ((-74.376292 40.531263, -74.374171 40....","POLYGON ((-74.26737 40.55216, -74.26719 40.552..."
1,34,013,009000,1,340130090001,1500000US340130090001,Block Group 1,G5030,S,395369,...,6.355383,50.229008,65.841161,15.441860,0.395369,3849.568378,13,NEWARK BROAD ST,"POLYGON ((-74.234371 40.769018, -74.231641 40....","POLYGON ((-74.18521 40.75686, -74.18516 40.757..."
2,34,013,010000,2,340130100002,1500000US340130100002,Block Group 2,G5030,S,291943,...,11.137163,8.017493,67.218282,9.833024,0.291943,4881.089802,17,BRICK CHURCH,"POLYGON ((-74.308114 40.810667, -74.307015 40....","POLYGON ((-74.22161 40.77576, -74.2166 40.7761..."
3,34,013,010000,2,340130100002,1500000US340130100002,Block Group 2,G5030,S,291943,...,11.137163,8.017493,67.218282,9.833024,0.291943,4881.089802,80,WATSESSING AVENUE,"POLYGON ((-74.260957 40.79571, -74.258466 40.7...","POLYGON ((-74.22161 40.77576, -74.2166 40.7761..."
4,34,013,021602,2,340130216022,1500000US340130216022,Block Group 2,G5030,S,1858617,...,0.000000,15.810277,71.536287,30.909091,1.858617,755.400386,102,LITTLE FALLS,"POLYGON ((-74.330102 40.876497, -74.330877 40....","POLYGON ((-74.26173 40.86661, -74.26071 40.867..."
5,34,003,042500,1,340030425001,1500000US340030425001,Block Group 1,G5030,S,1939493,...,4.388298,2.424242,71.619048,27.205882,1.939493,674.918651,70,NEW BRIDGE LANDING,"POLYGON ((-74.143244 40.954862, -74.143012 40....","POLYGON ((-74.07265 40.91087, -74.07255 40.911..."
6,34,003,037100,3,340030371003,1500000US340030371003,Block Group 3,G5030,S,520309,...,6.175299,12.715517,65.449804,35.754986,0.520309,1960.373547,69,RIVER EDGE,"POLYGON ((-74.123944 40.933874, -74.12336 40.9...","POLYGON ((-74.02867 40.93495, -74.02853 40.935..."
7,34,003,037100,3,340030371003,1500000US340030371003,Block Group 3,G5030,S,520309,...,6.175299,12.715517,65.449804,35.754986,0.520309,1960.373547,83,ORADELL,"POLYGON ((-74.097338 40.991148, -74.0966 40.98...","POLYGON ((-74.02867 40.93495, -74.02853 40.935..."
8,34,003,037100,4,340030371004,1500000US340030371004,Block Group 4,G5030,S,319796,...,9.682081,12.800000,70.468432,17.503587,0.319796,3940.011757,69,RIVER EDGE,"POLYGON ((-74.123944 40.933874, -74.12336 40.9...","POLYGON ((-74.02152 40.94039, -74.02109 40.941..."
9,34,003,037201,2,340030372012,1500000US340030372012,Block Group 2,G5030,S,459209,...,10.268949,2.602740,75.322284,32.471910,0.459209,3068.319654,69,RIVER EDGE,"POLYGON ((-74.123944 40.933874, -74.12336 40.9...","POLYGON ((-74.02621 40.93316, -74.0262 40.9332..."


## Calculating Demographic Weighted Averages by Station

This section computes **area-weighted averages** of key demographic variables for each NJ Transit station based on populations living within the 15-minute walking isochrones. This provides a more accurate representation of the socioeconomic context accessible by foot to each station.

---

### Steps Overview

1. **Specify Demographic Variables of Interest**
   - The following variables are selected for weighted averaging:
     - `Popoulation_Density`
     - `Median_Household_Income`
     - `Percent_Bach_Degree`
     - `Labor_Force_Participation_Rate`
     - `Employment_Rate`
     - `Unemployment_Rate`
     - `Percent_Public_Transit_Users`
     - `Median_Age`

2. **Initialize Data Storage**
   - Create an empty list called `weighted_averages` to collect results for each station.

3. **Group by Station**
   - Loop through each station's subset of intersected block groups in `intersection_gdf`.

4. **Filter and Calculate Weighted Averages**
   - For each demographic column:
     - Exclude invalid values (negative, zero, or `NaN`) from the block group data.
     - Compute the weighted average using block group area (`ALAND_km2`) as the weight:
       \[
       \text{Weighted Average} = \frac{\sum \text{Value} \times \text{Area}}{\sum \text{Area}}
       \]

5. **Store Results**
   - Append the computed averages for each station as a dictionary to the `weighted_averages` list.

6. **Create Final DataFrame**
   - Convert the list of dictionaries into a new `DataFrame` called `weighted_averages_df`.

---

This analysis results in a station-level dataset where each row includes aggregated demographic indicators of the population within walking distance of a transit station. These weighted features are useful for predicting ridership and understanding spatial equity in transit access.


In [10]:
# Define the columns to calculate weighted averages for
columns_to_average = [
    'Popoulation_Density', 'Median_Household_Income', 'Percent_Bach_Degree', 'Labor_Force_Participation_Rate',
    'Employment_Rate', 'Unemployment_Rate', 'Percent_Public_Transit_Users', 'Median_Age'
]

# Create an empty DataFrame to store weighted average results for each station
weighted_averages = []

# Group the data by 'station_name' to calculate weighted averages for each station
for station, group in intersection_gdf.groupby('station_name'):
    station_data = {'station_name': station}

    # Loop through each column to calculate the weighted average
    for column in columns_to_average:
        # Filter out invalid values (negative, zero, or NaN)
        valid_data = group[group[column] > 0].dropna(subset=[column, 'ALAND_km2'])

        if not valid_data.empty:
            # Calculate the weighted average
            weighted_avg = (valid_data[column] * valid_data['ALAND_km2']).sum() / valid_data['ALAND_km2'].sum()
        else:
            # If there are no valid values, set the weighted average as NaN
            weighted_avg = None

        # Add the weighted average to the station's data
        station_data[column + '_weighted_avg'] = weighted_avg

    # Append the station's data to the results list
    weighted_averages.append(station_data)

# Convert the list of results to a DataFrame
weighted_averages_df = pd.DataFrame(weighted_averages)

# Display the weighted averages DataFrame
print(weighted_averages_df)

         station_name  Popoulation_Density_weighted_avg  \
0    ABERDEEN-MATAWAN                       1485.254561   
1           ALLENDALE                        880.784940   
2          ALLENHURST                        814.240366   
3     ANDERSON STREET                       3668.155834   
4           ANNANDALE                        143.946569   
..                ...                               ...   
133          WESTWOOD                       2080.265941   
134       WHITE HOUSE                        133.722694   
135        WOOD-RIDGE                        763.013409   
136        WOODBRIDGE                       1791.412444   
137    WOODCLIFF LAKE                       1081.311102   

     Median_Household_Income_weighted_avg  Percent_Bach_Degree_weighted_avg  \
0                           100392.348558                         23.850899   
1                           164338.342100                         43.215385   
2                            91947.543687             

In [11]:
Data = pd.merge(Data, weighted_averages_df, left_on = 'Official NJT Station Name', right_on = 'station_name')

In [12]:
Data.to_csv("Data_With_IsoChrones.csv")